In [1]:
import pyscf
from pyscf import gto, scf, ao2mo
import numpy as np
import tequila as tw
import openfermion
import random
import math

from openfermionpyscf import run_pyscf
from openfermion import MolecularData, get_fermion_operator, InteractionOperator, get_interaction_operator
from openfermion import QubitOperator, taper_off_qubits, commutator, FermionOperator
from openfermion.transforms import get_fermion_operator, bravyi_kitaev, jordan_wigner

import scipy
from scipy.optimize import minimize
from scipy.linalg import expm
import copy

In [2]:
##A function that takes in molecular geometry, the basis, multiplicity, charge, and potentially
##a quibit transformation, and returns the created molecule along with it's Hamiltonian
def get_Hamiltonian(geometry, basis, multiplicity, charge, qubit_transf=None):
    mol = MolecularData(geometry=geometry, basis=basis, multiplicity=multiplicity, charge=charge)
    mol = run_pyscf(mol, run_scf=True, run_fci=True)
    ham = mol.get_molecular_hamiltonian()
    hamf = get_fermion_operator(ham)

    if(qubit_transf is None):
        return hamf, mol
    elif(qubit_transf == 'jw'):
        return jordan_wigner(hamf), mol
    elif(qubit_transf == 'bk'):
        return bravyi_kitaev(hamf), mol
    else:
        raise ValueError("qubit_transf must be None, 'jw', or 'bk'")

In [3]:
def convert_Hamiltonian(Ham, obj):
    Ham = copy.deepcopy(Ham)
    
    if obj == "interaction":
        return get_interaction_operator(Ham)

    if obj == "fermi":
        if hasattr(Ham, "terms"):
            return Ham
        return get_fermion_operator(Ham)

    if obj in ["jw", "bk"]:
        if not hasattr(Ham, "terms"):
            Ham = get_fermion_operator(Ham)

        if obj == "jw":
            return jordan_wigner(Ham)

        if obj == "bk":
            return bravyi_kitaev(Ham)

    raise ValueError("obj must be 'interaction', 'fermi', 'jw', or 'bk'")

In [4]:
##A fuction that generates an anti-Hermitian matrix from the indices in it's upper triangle
def generate_Kappa(dim, vals):
    if(len(vals) != (dim*dim)):
        raise ValueError(f"Expected {dim**2} real parameters, got {len(dim)}.")
    kappa = np.zeros((dim,dim),dtype = complex)
    index = 0

    for i in range(dim):
        kappa[i, i] = 1j * vals[index]
        index += 1

    for i in range(dim):
        for j in range(i + 1, dim):
            a = vals[index]
            b = vals[index + 1]

            kappa[i, j] = a + 1j*b
            kappa[j, i] = -1*(a + 1j*b).conjugate()

            index += 2
    return kappa

In [5]:
##A function that takes in a square matrix, and a threshold value and generates an orbital rotation
def unitary_Trasf(kappa, thresh=1e-12, qubit_transf=None):
    ##Get the dimensions of the Matrix
    n_spin_orbitals = kappa.shape[0]
    ##Returns the identity fermionic operator
    kappa_hat = FermionOperator.zero()

    for j in range(n_spin_orbitals):
        for i in range(n_spin_orbitals):
            ##Set the coefficient to match the matrix
            coeff = kappa[j, i]

            ##If the coefficients aren't tiny, add the fermionic operator
            if abs(coeff) > 1e-12:
                term = FermionOperator(((j, 1), (i, 0)), coeff)
                kappa_hat += term
                
    if(qubit_transf is None):
        return kappa_hat
        
    elif(qubit_transf == 'jw'):
        return jordan_wigner(kappa_hat)
        
    elif(qubit_transf == 'bk'):
        return bravyi_kitaev(kappa_hat)

In [6]:
##We can use the fact that the transformation is only dependent on the matrix to simplify this a great deal.
def rotate_Hamiltonian(Ham, kappa):
    ##Exponentiate Kappa
    U = expm(kappa)
    ##Rotate the Hamiltonian with the exponential
    ham_rot = copy.deepcopy(Ham)
    ham_rot.rotate_basis(U)
    ##Return the result
    return ham_rot

In [7]:
##Generate the Gammas
##This function takes in a set of sets G, where each g in G represents a subset of the spin-orbital indices
##It outputs a list of "gammas" that contain the information of which sets in G that orbital p is a part of
def build_gammas(G, mol):
    gammas = []
    m = mol.n_qubits
    for p in range(m):
        gamma_p = []
        for g in G:
            if(p in g):
                gamma_p.append(1)
            else:
                gamma_p.append(0)
        gammas.append(gamma_p)

    return np.array(gammas, dtype=int)

##Takes in a list of gammas and returns two sets of valid terms in the Hamiltonian (one body and two body terms)
def get_Valid_Terms(gammas):
    one_body_terms = set()
    two_body_terms = set()

    m = len(gammas)

    for i in range(m):
        for j in range(m):
            if np.array_equal(gammas[i], gammas[j]):
                one_body_terms.add((i, j))

    for p in range(m):
        for q in range(m):
            for r in range(m):
                for s in range(m):
                    if np.array_equal((gammas[p] + gammas[q]) % 2, (gammas[r] + gammas[s]) % 2):
                        two_body_terms.add((p, q, r, s))

    return one_body_terms, two_body_terms

In [8]:
##Takes in a Hamiltonian (or really any interacton operator), and a set of valid one and two body terms
##Outputs the Interaction Operator's terms that are present in the valid set
def cull_Hamiltonian(Ham, valid_one_body, valid_two_body):
    Ham_p = copy.deepcopy(Ham)

    m = Ham_p.one_body_tensor.shape[0]

    for p in range(m):
        for q in range(m):
            if (p,q) not in valid_one_body:
                Ham_p.one_body_tensor[p,q] = 0

    for p in range(m):
        for q in range(m):
            for r in range(m):
                for s in range(m):
                    if (p,q,r,s) not in valid_two_body:
                        Ham_p.two_body_tensor[p,q,r,s] = 0

    return Ham_p

In [9]:
##Takes in two Hamiltonians and returns the norm of their difference.
def Ham_Norm(H1, H2):
    diff = H1 - H2
    norm = math.sqrt((abs(diff.constant))**2 + np.sum((np.abs(diff.one_body_tensor))**2) + np.sum((np.abs(diff.two_body_tensor))**2))
    return(norm)

In [10]:
##I could have named this function better.  It takes in a set of parameters that define a unitary transformation,
##along with a Hamiltonian, and it returns the norm of the difference between a rotated Hamiltonian, and that same
##rotation projected onto the parent set.
def get_Norm(params, Ham, mol, doubles, quads, Ham_test=None):
    ##Generating the rotated Hamiltonian
    kappa = generate_Kappa(mol.n_qubits, params)
    Ham_r = rotate_Hamiltonian(Ham, kappa)

    if(Ham_test is not None):
        Ham_p = Ham_test
    else:
        Ham_p = cull_Hamiltonian(Ham_r, doubles, quads)
    
    return Ham_Norm(Ham_r, Ham_p)

In [ ]:
##Takes in a Hamiltonian, molecule, and potentially a test Hamiltonian to compare against.  It returns the optimized
##set of parameters that minimize the get_Norm function
def optimize_K(Ham, mol, groups, Ham_test=None, n_trial=1, rand_seed=0):

    np.random.seed(rand_seed)
    gammas = build_gammas(groups, mol)
    doubles, quads = get_Valid_Terms(gammas)

    n = mol.n_qubits
    x0 = np.zeros(n**2)
    x_list = [x0]
    if n_trial > 0:
        for _ in range(n_trial):
            xr = np.random.rand(n**2)
            x_list.append(xr)

    def objective(params):
        return get_Norm(params, Ham, mol, doubles, quads, Ham_test=Ham_test)

    print("Initial cost:", objective(x0))

    for i, x in enumerate(x_list):
        print("Trial: ", i)
        print("Initial cost:", objective(x))


        result = minimize(
            fun=objective,
            x0=x,
            method="L-BFGS-B",
            options={
                "maxiter": 500,
                "ftol": 1e-12,
                "gtol": 1e-8,
                "disp": False,
            }
        )
        print("Final cost:", result.fun)

        if i >0:
            if result.fun < min_result.fun:
                min_result = result
        else:
            min_result = result

    ##print("Success:", result.success)
    ##print("Message:", result.message)
    print("Final minimum cost:", min_result.fun)

    return min_result

In [18]:
##Takes in a Hamiltonian, molecule, and potentially a test Hamiltonian to compare against.  It returns the optimized
##set of parameters that minimize the get_Norm function
def optimize_K_double(Ham, mol, group1, group2, w1=1, w2=1, Ham_test=None, n_trial=1, rand_seed=0):

    np.random.seed(rand_seed)
    gammas1 = build_gammas(group1, mol)
    doubles1, quads1 = get_Valid_Terms(gammas1)

    gammas2 = build_gammas(group2, mol)
    doubles2, quads2 = get_Valid_Terms(gammas2)

    n = mol.n_qubits
    x0 = np.zeros(n**2)
    x_list = [x0]
    if n_trial > 0:
        for _ in range(n_trial):
            xr = np.random.rand(n**2)
            x_list.append(xr)

    def objective(params):
        return w1*get_Norm(params, Ham, mol, doubles1, quads1, Ham_test=Ham_test) + w2*get_Norm(params, Ham, mol, doubles2, quads2, Ham_test=Ham_test)

    print("Initial cost:", objective(x0))

    for i, x in enumerate(x_list):
        print("Trial: ", i)
        print("Initial cost:", objective(x))


        result = minimize(
            fun=objective,
            x0=x,
            method="L-BFGS-B",
            options={
                "maxiter": 500,
                "ftol": 1e-12,
                "gtol": 1e-8,
                "disp": False,
            }
        )
        print("Final cost:", result.fun)

        if i >0:
            if result.fun < min_result.fun:
                min_result = result
        else:
            min_result = result

    ##print("Success:", result.success)
    ##print("Message:", result.message)
    print("Final minimum cost:", min_result.fun)

    return min_result

In [ ]:
1.0 [Z0 Z3 Z4 Z7]
1.0 [Z1 Z3 Z5 Z7]
1.0 [Z2 Z3 Z6 Z7]
1.0 [Z1 Z2 Z4 Z7]
1.0 [Y0 Z1 Y2 Y4 Z5 Y6]
1.0 [Z7]
1.0 [X1 X2 X5 X6]
1.0 [Z0 Z5 Z6]

1.0 [Z0 Z3 Z4 Z7]
1.0 [Z1 Z3 Z5 Z7]
1.0 [Z2 Z3 Z6 Z7]
1.0 [Z1 Z2 Z4 Z7]
1.0 [Z1 Z3]
1.0 [Z1]
1.0 [Z0]
1.0 [Z7]

In [ ]:
##Hydrogen Tests:
geometry = [("H", (0.0, 0.0, 0.0)), ("H", (0.0, 0.0, 2.0)), ("H", (0.0, 0.0, 4.0)), ("H", (0.0, 0.0, 6.0))]
basis = "sto-3g"
multiplicity = 1
charge = 0
Ham, mol = get_Hamiltonian(geometry, basis, multiplicity, charge)
Ham = convert_Hamiltonian(Ham,"interaction")
candidate_groups_h2 = [
    [{0, 3, 4, 7}, {1, 3, 5, 7}, {2, 3, 6, 7}, {1, 2, 4, 7}],
    [{1, 3}, {1}, {0}, {7}],
]

x0 = np.zeros(mol.n_qubits**2)
##Calculating and printing the inital norm in equation 32 if the matrix is set to all zeros.
for groups in candidate_groups_h2:
    gammas = build_gammas(groups, mol)
    doubles, quads = get_Valid_Terms(gammas)
    cost = get_Norm(x0, Ham, mol, doubles, quads)

    result = optimize_K_double(Ham, mol, groups, n_trial=5)

    print(groups)
    print("Optimal Kappa: ", result.x)
    print()

##groups = [{0, 2}, {1, 3}]
##gammas = build_gammas(groups, mol)
##doubles, quads = get_Valid_Terms(gammas)

Initial cost: 0.4945200002893731
Trial:  0
Initial cost: 0.4945200002893731
Final cost: 0.47423338338441146
Trial:  1
Initial cost: 1.484863031414418
Final cost: 0.9761655691041071
Trial:  2
Initial cost: 1.5054195299257231
Final cost: 0.9487873198093678
Trial:  3
Initial cost: 1.5362573344553134
Final cost: 0.4742335453945002
Trial:  4
Initial cost: 1.4949463692939784
Final cost: 0.47423343793940664
Trial:  5
Initial cost: 1.5163740796259435
Final cost: 0.47423338338876103
Trial:  6
Initial cost: 1.5246406489029436
Final cost: 0.47423339823442134
Trial:  7
Initial cost: 1.4945048656550337
Final cost: 0.9104519098689249
Trial:  8
Initial cost: 1.4942530189105139
Final cost: 0.47423338826208333
Trial:  9
Initial cost: 1.4800415624355434
Final cost: 0.47423338386557917
Trial:  10
Initial cost: 1.5087938720842708
Final cost: 0.4742347063828517
Final cost: 0.47423338338441146
[{0, 3, 4, 7}, {1, 3, 5, 7}, {2, 3, 6, 7}, {1, 2, 4, 7}]
Optimal Kappa:  [-2.13935047e-09  0.00000000e+00  1.651895

In [ ]:

ops = [
    QubitOperator("Z0 Z2 Z4 Z7 Z9 Z10", 1.0),
    QubitOperator("Z1 Z3 Z5 Z7 Z9 Z11", 1.0),
    QubitOperator("Z6 Z7", 1.0),
    QubitOperator("Z8 Z9", 1.0),
    QubitOperator("Z0 Z2 Z4 Z6 Z10", 1.0),
    QubitOperator("Z6", 1.0),
    QubitOperator("Z0 Z4", 1.0),
    QubitOperator("Z1 Z5", 1.0),
    QubitOperator("Z10", 1.0),
    QubitOperator("Z11", 1.0),
    QubitOperator("Z1", 1.0),
    QubitOperator("Z0", 1.0),
]

In [33]:
##Hydrogen Tests:
geometry = [("Li", (0.0, 0.0, 0.0)), ("H", (0.0, 0.0, 1.6))]
basis = "sto-3g"
multiplicity = 1
charge = 0
Ham, mol = get_Hamiltonian(geometry, basis, multiplicity, charge)
Ham = convert_Hamiltonian(Ham,"interaction")
candidate_groups_h2 = [
    [{0, 1}, {2, 3}, {4, 5}, {6, 7}, {8, 9}, {10, 11}]
    # [{0, 2, 4, 7, 9, 10}, {1, 3, 5, 7, 9, 11}, {6, 7}, {8, 9}, {0, 2, 4, 6, 10}, {6}],
    # [{0, 4}, {1, 5}, {10}, {11}, {1}, {0}],
]

x0 = np.zeros(mol.n_qubits**2)
##Calculating and printing the inital norm in equation 32 if the matrix is set to all zeros.

result = optimize_K(Ham, mol, candidate_groups_h2[0], n_trial=5)

print(groups)
print("Optimal Kappa: ", result.x)
print()

##groups = [{0, 2}, {1, 3}]
##gammas = build_gammas(groups, mol)
##doubles, quads = get_Valid_Terms(gammas)

Initial cost: 0.7540507723510161
Trial:  0
Initial cost: 0.7540507723510161
Final cost: 0.6066480443500312
Trial:  1
Initial cost: 4.871210116053945
Final cost: 0.7322832681531255
Trial:  2
Initial cost: 4.356243917510466
Final cost: 0.7206888387875326
Trial:  3
Initial cost: 4.808908524314241
Final cost: 0.6928743886277667
Trial:  4
Initial cost: 4.780770311247966
Final cost: 0.872002686352237
Trial:  5
Initial cost: 5.075826957754148
Final cost: 0.6728852909647567
Final cost: 0.6066480443500312
[{0, 3, 4, 7}, {1, 3, 5, 7}, {2, 3, 6, 7}, {1, 2, 4, 7}]
Optimal Kappa:  [ 2.26125696e-08  2.62543286e-08 -7.68943796e-09 -7.56892211e-09
 -1.96947340e-08 -2.46576444e-08 -1.12681595e-09  0.00000000e+00
 -2.69574566e-10  0.00000000e+00  8.46223321e-09  1.88747774e-09
  1.19549312e-08  4.09626627e-08  2.41290248e-02 -5.80967331e-09
 -5.11292174e-09 -5.51608098e-09  3.73027308e-02 -5.43775855e-09
 -5.37804920e-09 -7.54078674e-09 -5.94297741e-09 -4.36274396e-09
 -5.94431797e-09 -5.80613946e-09 -4

In [34]:
from benchmark_all import benchmark_syms
from openfermion import get_ground_state, get_sparse_operator

kappa = generate_Kappa(mol.n_qubits, result.x)
H = Ham
HQ = jordan_wigner(H)
fci_e, fci_gs = get_ground_state(get_sparse_operator(HQ))

# ops_h4_1 = [
#     QubitOperator("Z0 Z3 Z4 Z7", 1.0),
#     QubitOperator("Z1 Z3 Z5 Z7", 1.0),
#     QubitOperator("Z2 Z3 Z6 Z7", 1.0),
#     QubitOperator("Z1 Z2 Z4 Z7", 1.0),
#     QubitOperator("Z1 Z3", 1.0),
#     QubitOperator("Z1", 1.0),
#     QubitOperator("Z0", 1.0),
#     QubitOperator("Z7", 1.0),
# ]

ops_lih_eqm = [
    QubitOperator("Z0 Z2 Z4 Z7 Z9 Z10", 1.0),
    QubitOperator("Z1 Z3 Z5 Z7 Z9 Z11", 1.0),
    QubitOperator("Z6 Z7", 1.0),
    QubitOperator("Z8 Z9", 1.0),
    QubitOperator("Z0 Z2 Z4 Z6 Z10", 1.0),
    QubitOperator("Z6", 1.0),
    QubitOperator("Z0 Z4", 1.0),
    QubitOperator("Z1 Z5", 1.0),
    QubitOperator("Z10", 1.0),
    QubitOperator("Z11", 1.0),
    QubitOperator("Z1", 1.0),
    QubitOperator("Z0", 1.0),
]

ops_lih_eqm = [
    QubitOperator("Z0 Z1", 1.0),
    QubitOperator("Z2 Z3", 1.0),
    QubitOperator("Z4 Z5", 1.0),
    QubitOperator("Z6 Z7", 1.0),
    QubitOperator("Z8 Z9", 1.0),
    QubitOperator("Z10 Z11", 1.0),
]

print("Unrotated")
benchmark_syms(ops_lih_eqm, HQ, fci_gs, fci_e, mol.n_qubits)

H = rotate_Hamiltonian(Ham, kappa)
HQ = jordan_wigner(H)
fci_e, fci_gs = get_ground_state(get_sparse_operator(HQ))

print("Rotated")
benchmark_syms(ops_lih_eqm, HQ, fci_gs, fci_e, mol.n_qubits)

Unrotated

Non commutative l1:  8.693999157458565
139/631 Terms in H found to commute with all symmetries.
Symmetries rotated to Z on qubits:  [0, 2, 4, 6, 8, 10]
Qubits permuted as:
0 -> 0
1 -> 6
2 -> 1
3 -> 7
4 -> 2
5 -> 8
6 -> 3
7 -> 9
8 -> 4
9 -> 10
10 -> 5
11 -> 11
Constructing unitary from factors and permutations...
Entropy of cuts (log base = e):
1 | 2 : 0.0005156174363906119
2 | 3 : 0.020980668000522746
3 | 4 : 0.05357467019857071
4 | 5 : 0.053574670198570755
5 | 6 : 0.05357467019857023
6 | 7 : 0.05126193438940142
7 | 8 : 0.05144768794522138
8 | 9 : 0.1291464534399854
9 | 10 : 0.0921052917493459
10 | 11 : 0.08836122119894167
11 | 12 : 0.08430845585500689
DMRG converged at bond dimension: 3
Rotated

Non commutative l1:  7.58232737034285
139/643 Terms in H found to commute with all symmetries.
Symmetries rotated to Z on qubits:  [0, 2, 4, 6, 8, 10]
Qubits permuted as:
0 -> 0
1 -> 6
2 -> 1
3 -> 7
4 -> 2
5 -> 8
6 -> 3
7 -> 9
8 -> 4
9 -> 10
10 -> 5
11 -> 11
Constructing unitary fro

BenchmarkData(tag='', symmetries=[1.0 [Z0 Z1], 1.0 [Z2 Z3], 1.0 [Z4 Z5], 1.0 [Z6 Z7], 1.0 [Z8 Z9], 1.0 [Z10 Z11]], non_commuting_l1=7.58232737034285, num_commuting_terms=139, sym_entropy=0, cut_entropies=[0.025796407618296397, 0.14757728119852295, 0.1465197169748445, 0.14651971697484467, 0.14651971697484253, 0.1327447572664592, 0.13293421680176834, 0.20062560107807242, 0.09467250754068354, 0.09093236221366781, 0.08687648505143389], dmrg_bd=4, single_sector_e=0)

In [64]:
##Lithium Tests:
geometry = [("Li", (0.0, 0.0, 0.0)), ("H", (0.0, 0.0, 1.6))]
basis = "sto-3g"
multiplicity = 1
charge = 0
Ham, mol = get_Hamiltonian(geometry, basis, multiplicity, charge)
Ham = convert_Hamiltonian(Ham,"interaction")
candidate_groups_lih = [
    [{0}, {1}, {2}, {3}, {4}],
    [{0}, {2}, {4}, {6}, {8}],
    [{1}, {3}, {5}, {7}, {9}],

    [{0, 1}, {2, 3}, {4, 5}, {6, 7}, {8, 9}, {10, 11}],

    [{0, 2, 4, 6, 8, 10, 12}, {1, 3, 5, 7, 9, 11}],

    [{0, 3}, {1, 2}],
    [{0, 5}, {1, 4}, {2, 7}, {3, 6}],
    [{0, 6}, {1, 7}, {2, 8}, {3, 9}],

    [{0}, {1}, {10}, {11}],
    [{0}, {1}, {2}, {3}, {8}, {9}, {10}, {11}]
]

x0 = np.zeros(mol.n_qubits**2)
##Calculating and printing the inital norm in equation 32 if the matrix is set to all zeros.
for groups in candidate_groups_lih:
    gammas = build_gammas(groups, mol)
    doubles, quads = get_Valid_Terms(gammas)
    cost = get_Norm(x0, Ham, mol, doubles, quads)

    result = optimize_K(Ham, mol, groups)

    print(groups)
    kappa = result.x
    for i in range(len(kappa)):
        if kappa[i] < 1e-7:
            kappa[i] = 0
    print("Optimal Kappa: ", result.x)
    print()

Initial cost: 0.798145455056402
Final cost: 0.6650841699056199
[{0}, {1}, {2}, {3}, {4}]
Optimal Kappa:  [0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.0243097  0.         0.         0.
 0.03719473 0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.03636152 0.         0.         0.         0.0513514  0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.04638931 0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.05476703 0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.       